In [1]:
from langgraph.graph import StateGraph, START, END
from typing import Annotated, TypedDict
from langchain_core.messages import HumanMessage, BaseMessage
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages
from dotenv import load_dotenv

from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool

import requests
import random

C:\Users\richi\AppData\Local\Temp\ipykernel_1500\3122456501.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [2]:
load_dotenv()

True

In [3]:
llm = ChatOpenAI(model='gpt-4o-mini')

In [4]:
search_tool = DuckDuckGoSearchRun(region='us-en')

@tool
def calculator(first_number, second_number, operation):
    """
    Perform a basic arithmetic operation on two numbers.
    Supported operations: add, sub, mul, div
    """
    try:
        if operation == "sum":
            result = first_number + second_number

        elif operation == "sub":
            result = first_number - second_number

        elif operation == "mul":
            result = first_number * second_number

        elif operation == "div":
            result = first_number / second_number

        else:
            raise ValueError("Invalid operation")

        return {
            "first_number": first_number,
            "second_number": second_number,
            "operation": operation,
            "result": result
        }

    except Exception as e:
        return {
            "first_number": first_number,
            "second_number": second_number,
            "operation": operation,
            "result": f"Error: {str(e)}"
        }
        
@tool
def get_stock_price(symbol: str)-> dict:
    """
    Fetch latest stock price for a given symbol (e.g. 'AAPL', 'TSLA')
    using Alpha Vantage with API key in the URL
    """
    url = "https://api.twelvedata.com/quote"
    params = {
        "symbol": symbol.upper(),
        "apikey": "56033ee4874447edbe330896b1eac718"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    return response.json()

In [5]:
tools = [get_stock_price, search_tool, calculator]

llm_with_tools = llm.bind_tools(tools)

In [6]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [7]:
def chat_node(state: ChatState):
    """LLM node that may answer or request a tool call"""
    message = state['messages']
    response = llm_with_tools.invoke(message)
    return {'messages': response}

tool_node = ToolNode(tools=tools)

In [8]:
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)

In [9]:
graph.add_edge(START, "chat_node")

graph.add_conditional_edges("chat_node", tools_condition)

graph.add_edge("tools", "chat_node")

In [10]:
chatbot = graph.compile()

In [11]:
out = chatbot.invoke({'messages': [HumanMessage(content="What is 2*3")]})
print(out['messages'][-1].content)

The result of \( 2 \times 3 \) is 6.


In [12]:
out = chatbot.invoke({'messages': [HumanMessage(content="What is current stock price of Apple")]})
print(out['messages'][-1].content)

The current stock price of Apple Inc. (AAPL) is $334.62. Here are some additional details:

- **Open:** $327.49
- **High:** $334.90
- **Low:** $326.57
- **Previous Close:** $326.57
- **Change:** $8.05 (2.47% increase)
- **Volume:** 486,045 shares traded
- **Market Status:** Market is currently open. 

The stock has a 52-week range of $226.65 to $344.57.


In [13]:
out = chatbot.invoke({'messages': [HumanMessage(content="What is current stock price of Apple. How much it will cost to purchase 50 shares of it?")]})
print(out['messages'][-1].content)

The current stock price of Apple Inc. (AAPL) is $334.62. To purchase 50 shares, it would cost you $16,731.00.
